# Tech Challenge - Fase 1: Predição de AVC

**Autor:** Gustavo Diniz - Pós-Tech FIAP, Tech Challenge.

**Problema.** Um hospital universitário recebe um volume alto de pacientes e quer um sistema de apoio à triagem que sinalize quem está em maior risco de Acidente Vascular Cerebral (AVC). O médico continua com a palavra final, mas a saída do modelo serve como alerta para priorizar atendimento.

**Dataset.** *Stroke Prediction Dataset* (autor: fedesoriano, Kaggle). Contém 5.110 registros e 11 atributos clínicos por paciente (idade, hipertensão, doença cardíaca, IMC, glicose média, tabagismo etc.) e a variável alvo `stroke`. O dataset é fortemente desbalanceado (cerca de 4,9% de casos positivos), o que terá impacto direto na escolha das métricas de avaliação.

**Roteiro do notebook.**
1. Carga e EDA
2. Pré-processamento e análise de correlação
3. Modelagem (Regressão Logística, Árvore de Decisão, Random Forest) com separação treino/validação/teste
4. Avaliação no conjunto de teste
5. Interpretabilidade (feature importance + SHAP) e discussão crítica

## 1. Setup e carga do dataset

**Pré-requisito:** baixar o CSV do Kaggle e colocar em `../data/healthcare-dataset-stroke-data.csv`.

Passos (uma vez só):
1. Abrir https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset
2. Extrair o `archive.zip` e copiar o `healthcare-dataset-stroke-data.csv` para a pasta `data/` deste projeto.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

In [ ]:
CSV_PATH = '../data/healthcare-dataset-stroke-data.csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'Arquivo não encontrado em {CSV_PATH}. '
        'Baixe o CSV do Kaggle '
        '(https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset) '
        'e coloque-o na pasta data/.'
    )

df = pd.read_csv(CSV_PATH)
# No CSV original, BMI vem com 'N/A' em valores ausentes; força conversão para numérico.
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')

print('Shape:', df.shape)
df.head()

## 2. Visão geral das colunas

Antes de qualquer análise, vale entender o que cada coluna significa:

- `id` — identificador único do paciente. Não tem valor preditivo, vamos descartar.
- `gender` — Male, Female, Other.
- `age` — idade em anos.
- `hypertension` — 1 se o paciente tem hipertensão, 0 caso contrário.
- `heart_disease` — 1 se tem doença cardíaca, 0 caso contrário.
- `ever_married` — Yes/No.
- `work_type` — Private, Self-employed, Govt_job, children, Never_worked.
- `Residence_type` — Urban/Rural.
- `avg_glucose_level` — nível médio de glicose no sangue.
- `bmi` — índice de massa corporal.
- `smoking_status` — formerly smoked, never smoked, smokes, Unknown.
- `stroke` — alvo: 1 se o paciente teve AVC, 0 caso contrário.

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.to_frame('qtd_ausentes').assign(pct=lambda x: (x['qtd_ausentes'] / len(df) * 100).round(2))

A única coluna com valores faltantes é `bmi` (cerca de 4%). Vamos tratar por imputação pela mediana na etapa de pré-processamento, já que IMC tem distribuição assimétrica.

## 3. Estatísticas descritivas

In [ ]:
df.describe().T.round(2)

In [ ]:
df.describe(include='object').T

Observações iniciais:

- `age` vai de aproximadamente 0,08 a 82 anos.
- `avg_glucose_level` tem amplitude grande (cerca de 55 a 272 mg/dL), sugerindo pacientes diabéticos na base.
- `bmi` tem média perto de 28 (sobrepeso).
- `gender` tem 3 categorias, mas 'Other' aparece em pouquíssimos casos.

In [ ]:
for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
    print(col)
    print(df[col].value_counts(dropna=False))
    print('-' * 40)

Como esperado, `gender = Other` aparece em apenas 1 registro — vamos descartar essa linha para evitar problema na divisão estratificada.

Em `smoking_status`, a categoria `Unknown` é considerável (cerca de 30%). Vamos manter como uma categoria própria em vez de tentar imputar.

## 4. Distribuição da variável alvo

In [ ]:
stroke_counts = df['stroke'].value_counts()
stroke_pct = df['stroke'].value_counts(normalize=True) * 100
print(stroke_counts)
print()
print(stroke_pct.round(2))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x='stroke', ax=ax)
ax.set_title('Distribuição da variável alvo (stroke)')
ax.set_xlabel('stroke (0 = não, 1 = sim)')
plt.show()

Dataset fortemente desbalanceado (menos de 5% positivos). Três decisões práticas:

1. **Acurácia engana**: um modelo que sempre prediz 0 acerta ~95% sem aprender nada. A métrica principal será **recall** na classe positiva, complementada por **F1-score** e **AUC-ROC**.
2. **Lidar com o desbalanceamento no treino**: vamos usar `class_weight='balanced'`.
3. **Estratificação obrigatória** na divisão treino/validação/teste.

## 5. Distribuições univariadas

In [ ]:
num_cols = ['age', 'avg_glucose_level', 'bmi']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(df[col].dropna(), bins=40, ax=ax, kde=True)
    ax.set_title(f'Distribuição de {col}')
plt.tight_layout()
plt.show()

Comentários:

- `age` quase uniforme, com leve pico após os 50 anos.
- `avg_glucose_level` é bimodal (segundo pico acima de 180 mg/dL, possíveis diabéticos).
- `bmi` é assimétrica à direita — mediana é melhor que média para imputação.

In [ ]:
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status',
            'hypertension', 'heart_disease']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
axes.flatten()[-1].axis('off')
plt.tight_layout()
plt.show()

## 6. Padrões bivariados em relação a AVC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.kdeplot(data=df, x=col, hue='stroke', common_norm=False, ax=ax)
    ax.set_title(f'{col} por classe alvo')
plt.tight_layout()
plt.show()

Observações:

- **Idade é o sinal mais forte.** Pacientes com AVC se concentram acima dos 60 anos.
- **Glicose** está deslocada à direita no grupo com AVC (cauda densa em níveis altos).
- **BMI** mostra pouca separação entre os grupos.

In [ ]:
for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status',
            'hypertension', 'heart_disease']:
    tab = df.groupby(col)['stroke'].agg(['count', 'sum', 'mean'])
    tab['taxa_avc_%'] = (tab['mean'] * 100).round(2)
    tab = tab.rename(columns={'count': 'n', 'sum': 'casos_avc'}).drop(columns='mean')
    print(col)
    print(tab)
    print('-' * 50)

Resumo das taxas:

- `heart_disease=1`: taxa de AVC ~17% (4x maior).
- `hypertension=1`: taxa de AVC ~13%.
- `ever_married=Yes`: taxa alta, mas provavelmente confundida com idade.
- `work_type=children`: taxa quase zero (efeito de idade).
- `Residence_type` e `gender`: pouca diferença.

# Etapa 2 - Pré-processamento e análise de correlação

In [ ]:
# Limpeza inicial:
# - Remove a linha com gender='Other' (apenas 1 registro)
# - Descarta a coluna id (sem valor preditivo)
df_clean = df[df['gender'] != 'Other'].drop(columns=['id']).reset_index(drop=True)
print('Antes:', df.shape, '| Depois:', df_clean.shape)

## Análise de correlação

Para variáveis numéricas, uso Pearson. Para incluir categóricas na mesma matriz, faço primeiro uma codificação numérica simples (apenas para visualização).

In [ ]:
num_for_corr = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'stroke']
corr_num = df_clean[num_for_corr].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_num, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlação - variáveis numéricas')
plt.show()

In [ ]:
# Matriz de correlação completa, com categóricas codificadas (apenas para visualização)
df_corr = df_clean.copy()
for col in df_corr.select_dtypes(include='object').columns:
    df_corr[col] = pd.Categorical(df_corr[col]).codes

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlação - todas as features (categóricas codificadas)')
plt.show()

Pontos a destacar da matriz:

- A maior correlação com `stroke` é com `age` (~0,25), seguida por `heart_disease` (~0,13), `hypertension` (~0,13) e `avg_glucose_level` (~0,13).
- `age` e `ever_married` têm correlação alta entre si (~0,68), o que confirma a suspeita do confundimento na EDA — vamos manter as duas e deixar o modelo decidir.
- Não existem variáveis quase perfeitamente correlacionadas entre si, então não precisamos remover features por multicolinearidade extrema.

## Separação treino / validação / teste

O desafio pede separação clara entre treino, validação e teste. Vou usar a divisão:

- **Treino**: 64% (treino do modelo)
- **Validação**: 16% (escolha de modelo e ajuste de hiperparâmetros)
- **Teste**: 20% (avaliação final, intocado até o fim)

Tudo estratificado pela variável alvo.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['stroke'])
y = df_clean['stroke']

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=RANDOM_STATE
)

print(f'Treino:    {X_train.shape} | positivos: {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Validação: {X_val.shape} | positivos: {y_val.sum()} ({y_val.mean()*100:.2f}%)')
print(f'Teste:     {X_test.shape} | positivos: {y_test.sum()} ({y_test.mean()*100:.2f}%)')

## Pipeline de pré-processamento

Usando `ColumnTransformer` para aplicar transformações diferentes a colunas diferentes, dentro de um pipeline do scikit-learn (evita data leakage e simplifica o deploy):

- Numéricas (`age`, `avg_glucose_level`, `bmi`): imputação pela mediana + `StandardScaler`.
- Categóricas (`gender`, `ever_married`, `work_type`, `Residence_type`, `smoking_status`): `OneHotEncoder`.
- Binárias (`hypertension`, `heart_disease`): passam direto.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
binary_features = ['hypertension', 'heart_disease']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('bin', 'passthrough', binary_features),
])

X_train_t = preprocessor.fit_transform(X_train)
print('Shape após pré-processamento:', X_train_t.shape)
feature_names = (
    numeric_features
    + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
    + binary_features
)
print('Features geradas:', feature_names)

# Etapa 3 - Modelagem

Vou treinar três modelos cobrindo famílias diferentes vistas nas aulas:

1. **Regressão Logística** — modelo linear, fácil de interpretar.
2. **Árvore de Decisão** — modelo baseado em divisões, captura não-linearidades.
3. **Random Forest** — ensemble de árvores, tende a ser o mais robusto.

Todos com `class_weight='balanced'` para compensar o desbalanceamento.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Regressão Logística': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Árvore de Decisão': DecisionTreeClassifier(
        class_weight='balanced', max_depth=6, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, class_weight='balanced', max_depth=10,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
}

pipelines = {
    name: Pipeline([('preprocessor', preprocessor), ('model', model)])
    for name, model in models.items()
}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    print(f'Treinado: {name}')

## Validação cruzada no conjunto de treino

Antes de olhar para a validação fixa, faço também uma validação cruzada estratificada para checar estabilidade dos resultados.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = []
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({
        'modelo': name,
        'auc_medio': scores.mean(),
        'auc_std': scores.std(),
    })

pd.DataFrame(cv_results).round(4)

## Avaliação no conjunto de validação

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(pipe, X, y):
    y_pred = pipe.predict(X)
    y_proba = pipe.predict_proba(X)[:, 1]
    return {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred),
        'f1': f1_score(y, y_pred),
        'roc_auc': roc_auc_score(y, y_proba),
    }

val_results = {name: evaluate(pipe, X_val, y_val) for name, pipe in pipelines.items()}
val_df = pd.DataFrame(val_results).T.round(4)
val_df

## Escolha do modelo final

Considerando que **recall** é a métrica mais importante (queremos não deixar passar pacientes com risco de AVC), e usando F1/AUC como critério de desempate, o modelo escolhido é o que apresentar melhor recall com bom AUC.

In [ ]:
best_model_name = val_df['recall'].idxmax()
best_pipe = pipelines[best_model_name]
print(f'Modelo selecionado: {best_model_name}')
val_df.loc[[best_model_name]]

# Etapa 4 - Avaliação no conjunto de teste

Agora avalio o modelo escolhido no conjunto de teste, que não foi tocado até aqui.

In [ ]:
test_metrics = evaluate(best_pipe, X_test, y_test)
pd.Series(test_metrics).round(4).to_frame(f'{best_model_name} - teste')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve

y_pred_test = best_pipe.predict(X_test)
y_proba_test = best_pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_test, target_names=['não AVC', 'AVC']))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['não AVC', 'AVC']).plot(ax=ax, colorbar=False)
ax.set_title(f'Matriz de confusão - {best_model_name} (teste)')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_test)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'{best_model_name} (AUC={test_metrics["roc_auc"]:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Aleatório')
ax.set_xlabel('Falso positivo')
ax.set_ylabel('Verdadeiro positivo')
ax.set_title('Curva ROC - conjunto de teste')
ax.legend()
plt.show()

## Discussão da métrica escolhida

Para um problema de triagem de AVC, o erro mais grave é o **falso negativo** (deixar passar um paciente que terá AVC). Por isso priorizo **recall** na classe positiva.

A acurácia, isoladamente, é enganosa nesse contexto: um modelo trivial que prevê sempre `stroke=0` atingiria cerca de 95% de acurácia sem nenhuma utilidade clínica. **F1** ajuda a balancear precision e recall, e **AUC-ROC** mede a separabilidade independente do threshold. As três métricas juntas dão um retrato mais honesto do modelo.

# Etapa 5 - Interpretabilidade

O médico precisa entender por que o modelo sinalizou um paciente. Aqui uso duas abordagens complementares:

1. **Feature importance global** (do próprio modelo, se for baseado em árvore; ou coeficientes da regressão logística).
2. **SHAP values** — explicam contribuições por instância e ajudam a entender direção dos efeitos.

In [ ]:
preproc = best_pipe.named_steps['preprocessor']
feat_names = (
    numeric_features
    + list(preproc.named_transformers_['cat'].get_feature_names_out(categorical_features))
    + binary_features
)

model = best_pipe.named_steps['model']

if hasattr(model, 'feature_importances_'):
    importances = pd.Series(model.feature_importances_, index=feat_names).sort_values(ascending=False)
    title = 'Feature importance (modelo baseado em árvore)'
elif hasattr(model, 'coef_'):
    importances = pd.Series(np.abs(model.coef_[0]), index=feat_names).sort_values(ascending=False)
    title = 'Importância pelo valor absoluto do coeficiente (Regressão Logística)'
else:
    importances = None

if importances is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    importances.head(15).plot(kind='barh', ax=ax)
    ax.invert_yaxis()
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    print(importances.head(10).to_frame('importancia'))

In [ ]:
import shap

X_test_t = best_pipe.named_steps['preprocessor'].transform(X_test)
X_test_t_df = pd.DataFrame(X_test_t, columns=feat_names)

if isinstance(model, (DecisionTreeClassifier, RandomForestClassifier)):
    sample = X_test_t_df.sample(min(300, len(X_test_t_df)), random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)
    if isinstance(shap_values, list):
        sv_pos = shap_values[1]
    elif shap_values.ndim == 3:
        sv_pos = shap_values[..., 1]
    else:
        sv_pos = shap_values
    shap.summary_plot(sv_pos, sample, show=True)
elif isinstance(model, LogisticRegression):
    sample = X_test_t_df.sample(min(300, len(X_test_t_df)), random_state=RANDOM_STATE)
    explainer = shap.LinearExplainer(model, X_test_t_df)
    shap_values = explainer.shap_values(sample)
    shap.summary_plot(shap_values, sample, show=True)

## Como ler o gráfico SHAP

- Cada ponto representa um paciente do conjunto de teste.
- Eixo X (SHAP value): contribuição daquela feature para empurrar a predição para AVC (direita) ou não-AVC (esquerda).
- Cor: valor da feature (vermelho = alto, azul = baixo).

Os padrões esperados (e geralmente confirmados pelo gráfico) são:

- **age**: vermelho à direita (idade alta empurra para AVC).
- **avg_glucose_level**: vermelho à direita (glicose alta empurra para AVC).
- **hypertension** e **heart_disease**: presença empurra para AVC.

Isso é coerente com o que se sabe clinicamente sobre fatores de risco de AVC.

# Discussão crítica

**O modelo pode ser usado na prática?** Como **ferramenta de apoio**, sim. Como ferramenta de decisão autônoma, não.

Pontos a favor:
- O recall na classe positiva está em patamar útil, ou seja, o modelo identifica a maior parte dos casos verdadeiros.
- O AUC mostra que existe separabilidade real entre as classes.
- As features mais importantes batem com o conhecimento médico (idade, hipertensão, doença cardíaca, glicose), o que dá confiança de que o modelo não está se apoiando em ruído.

Pontos contra:
- **Precision baixa**: o custo é gerar muitos falsos positivos, ou seja, encaminhar pacientes sem AVC para avaliação adicional. Isso é aceitável se o objetivo é triagem, mas precisa ser comunicado claramente.
- **Base pequena de positivos**: apenas ~250 pacientes com AVC no dataset todo.
- **Vieses do dataset**: a coleta é específica e não reflete necessariamente a população do hospital onde o modelo seria usado. Antes de qualquer deploy real, seria necessário recalibrar com dados do próprio hospital.

**Como usaria na prática:**

1. Como **alerta** no prontuário: "este paciente está acima do limiar de risco do modelo (probabilidade X%) — considerar avaliação adicional".
2. Sempre acompanhado da explicação SHAP, para que o médico veja **quais fatores** levaram àquela predição.
3. O médico tem a palavra final. O modelo não substitui exame clínico, anamnese, nem decisão terapêutica.
4. Monitorar continuamente recall e precision em produção, e re-treinar com dados novos a cada período (semestral, por exemplo).